In [ ]:
!pwd

In [ ]:
import numpy as np
import pandas as pd
import os
import scipy.stats
import matplotlib.pyplot as plt
from matplotlib.patches import Patch


os.chdir('/Users/zagidull/Desktop/synosys/lina-s paper/')

In [ ]:
df = pd.read_csv('data/SynoSys_27.02.2026-11.03.2026/bbi-(enhanced).csv')
df.head()

### Preprocess

In [ ]:
assert any(df.isna())

print(df.isna().sum())

In [ ]:
df = df.dropna(axis='columns', inplace=False) 
print(df.isna().sum())

In [ ]:
pd.DataFrame(map(lambda col: (col, df[col].nunique(), df[col].dtype ), df.columns))

# looks like these 'Group Names' and 'BBI Type' cols can be dropped. Lets check other columns next 

In [ ]:
all(
    pd.to_datetime(df['Start Time (ms)'], unit='ms', utc=True) \
    + pd.to_timedelta(df['Time Zone (s)'], unit='s') \
    == pd.to_datetime(df['Start Time (Local)'], utc=True))

# looks like Start Time (ms) is in UTC, ie Start Time (ms)+ Time Zone (s)= Start Time (Local)

In [ ]:
df = df.assign( timeLocal_ms = df['Start Time (ms)']+df['Time Zone (s)']*1000) # so lets transform 'Start Time (ms)' to local time (ms), and then drop all other time columns
df.head()

In [ ]:
 # sanity check just for the first row
assert pd.Timestamp(1_772_789_052_000, unit='ms')+pd.Timedelta(3_600*1000, unit='ms') == pd.Timestamp(1_772_792_652_000, unit='ms')
assert pd.Timestamp(1_772_792_652_000, unit='ms') == pd.to_datetime(df['Start Time (Local)'][0], utc=False)
print('ok')

In [ ]:
cols_drop=['User First Name', 'User Email', 'Group Names', 'Calendar Date (Local)', 'Start Time (Local)', 'Start Time (UTC)', 'Time Zone (Local)', 'Calendar Date (UTC)', 'BBI Type', 'Start Time (ms)', 'Time Zone (s)']
df = df.drop(labels=cols_drop, axis='columns', inplace=False)
df.head()

In [ ]:
# lets map user id to something simple, like 0-1-2

userid_map = dict(zip( list(df['User Id'].unique()), range(0, df['User Id'].nunique()))) # better than using set(), because sets are non-deterministic
df['User Id'] = df['User Id'].map(userid_map)


### Statistical EDA

In [ ]:
# how long measurements were collected for, stratify by User Id

df.groupby('User Id')['timeLocal_ms'].agg(lambda x: pd.to_timedelta( max(x)-min(x), unit='ms' ))

In [ ]:
# look at users

df.groupby('User Id')['BBI (ms)']\
    .agg(['mean','median','std','count']).round(0) 

# so id=0 has ~same mean&median
# 1 and 2 seem to skew to the right (median < mean), ie probably high-val outliers
# all have roughly equal std

In [ ]:
# look at status

df.groupby('Status')['BBI (ms)']\
    .agg(['mean','median' ,'std', 'count']).round(0)

# low-confidence scores have a much higher std than high confidence, and also right-skewed

In [ ]:
# look at users&confidence,
# calculate min/max+moments,
# compare with ref normal distributions

ref_norm = scipy.stats.norm.rvs(loc=df['BBI (ms)'].mean(), 
                                scale=df['BBI (ms)'].std(), 
                                size=df.shape[0],random_state=1)
f=lambda v: [np.min(v),
             np.max(v),
             np.mean(v), 
             np.median(v), 
             np.std(v), 
             scipy.stats.skew(v), 
             scipy.stats.kurtosis(v)]

# in stratified data
gr = df.groupby(['Status', 'User Id'])['BBI (ms)']\
    .agg(['min','max','mean', 'median', 'std', 'skew', pd.Series.kurt])

gr.loc[('','all'), :] = f(df['BBI (ms)']) # all data
gr.loc[('','ref normal'), :] = f(ref_norm) # ref dist

gr.style.background_gradient(subset=['min','max','skew','kurt'], cmap='coolwarm')\
    .format({k:'{:.0f}' for k in gr.columns})

### Visual EDA

In [ ]:
# half-violin comparison:
# left half  = actual BBI distribution
# right half = matched normal with sample mean/std

#q99 = df['BBI (ms)'].quantile(0.995)
plot_df = df.copy()
plot_df['BBI (ms)'] = (df.groupby(['User Id', 'Status'])['BBI (ms)'].transform(lambda s: s.clip(upper=s.quantile(0.999))))


fig, ax = plt.subplots(figsize=(9, 5))

labels = []
positions = []
pos = 1

for user in sorted(plot_df['User Id'].unique()):
    for status in sorted(plot_df['Status'].unique()):
        s = plot_df.loc[(plot_df['User Id'] == user) & (plot_df['Status'] == status), 'BBI (ms)'].to_numpy()
        normal_ref = scipy.stats.norm.rvs(loc=s.mean(), scale=s.std(ddof=1), size=s.shape[0], random_state=1)

        v_actual = ax.violinplot([s], positions=[pos], widths=0.8, showextrema=False)
        v_normal = ax.violinplot([normal_ref], positions=[pos], widths=0.8, showextrema=False)

        # keep only left half for actual data
        body_actual = v_actual['bodies'][0]
        verts = body_actual.get_paths()[0].vertices
        verts[:, 0] = np.minimum(verts[:, 0], pos)
        body_actual.set_facecolor('steelblue')
        body_actual.set_edgecolor('steelblue')
        body_actual.set_alpha(0.7)

        # keep only right half for matched normal
        body_normal = v_normal['bodies'][0]
        verts = body_normal.get_paths()[0].vertices
        verts[:, 0] = np.maximum(verts[:, 0], pos)
        body_normal.set_facecolor('crimson')
        body_normal.set_edgecolor('crimson')
        body_normal.set_alpha(0.5)

        # median lines
        med_actual = np.median(s)
        med_normal = np.median(normal_ref)
        ax.hlines(med_actual, pos - 0.18, pos, color='midnightblue', lw=2)
        ax.hlines(med_normal, pos, pos + 0.18, color='darkred', lw=2)

        labels.append(f'{status.split('_')[0]}\nUser {user}')
        positions.append(pos)
        pos += 1

    pos += 0.5

ax.set_xticks(positions)
ax.set_xticklabels(labels)
ax.set_ylabel('BBI [ms]')
ax.set_title('BBI distributions with medians: actual vs matched normal')
ax.grid(axis='y', alpha=0.2)

legend_handles = [
    Patch(facecolor='steelblue', edgecolor='steelblue', alpha=0.7, label='user data'),
    Patch(facecolor='crimson', edgecolor='crimson', alpha=0.5, label='ref normal'),
]
ax.legend(handles=legend_handles, frameon=False)

plt.tight_layout()
plt.show()


In [ ]:
# lets look at a single user and show their measurements, x-axis is ms from the start of the analysis
user=0

temp=df.loc[df['User Id']==user,:]
x=temp['timeLocal_ms']-temp['timeLocal_ms'].min() 
y=temp['BBI (ms)']

fig,ax=plt.subplots()
ax.scatter(x=x, y=y, marker='x', s=3)
ax.set_title(f'Interbeat intervals over the test period for id={user}')
ax.set_xlabel('Time from start [ms]')
ax.set_ylabel('BBI [ms]') 

# clearly ms is a shitty resolution + there are a few outliers that make BBI values squished

In [ ]:
# winsorize, moving average, and groupby time of day for a single user
user=0
upper_clip=0.999 # very modest 
w=3_000 # do rolling average over w window

fig,ax=plt.subplots(nrows=2, ncols=2, figsize=(18,12))
fig.suptitle(f'Interbeat intervals over the test period for id={user}')
temp=df.loc[df['User Id']==user,:]

# Plot 1. clipped, no other processing
x=pd.to_timedelta(temp['timeLocal_ms']-temp['timeLocal_ms'].min(), unit='ms').dt.total_seconds() /3600 # let's change x to better unit, eg hours?
q_h=temp['BBI (ms)'].quantile(upper_clip) 
q_l=temp['BBI (ms)'].quantile(1-upper_clip) 
y=temp['BBI (ms)'].clip(upper=q_h, lower=q_l)
ax[0,0].scatter(x=x, y=y, s=.05)
ax[0,0].set_xlabel('Time from start [hrs]')
ax[0,0].set_ylabel('cBBI [ms]')

# Plots 2&4. clipped + moving average
y=y.rolling(window=w, min_periods=1).mean()
for status in ['HIGH_CONFIDENCE', 'LOW_CONFIDENCE']:
    mask=temp['Status'] == status
    if status=='HIGH_CONFIDENCE': 
        a=ax[0,1] 
        c='tab:blue'
    else: 
        a=ax[1,1]
        c='tab:orange'
    a.scatter(x=x[mask], y=y[mask], label=status.split('_')[0], marker='+', s=5, c=c)
    a.legend()
    a.set_xlabel('Time from start [hrs]')
    a.set_ylabel('cBBI [ms] + rolling avg') 
    a.set_ylim((450,1250))
    

# Plot 3. groupby [hour_of_day, status], then clip. Add CIs
hr=pd.to_datetime(temp['timeLocal_ms'], unit='ms').dt.hour
y=temp.groupby([hr, 'Status'])['BBI (ms)'].agg(
    lambda s: np.mean(s.clip(upper=q_h, lower=q_l))
    ).unstack('Status')
y_int=temp.groupby([hr, 'Status'])['BBI (ms)'].agg(
    lambda s: np.std(s.clip(upper=q_h, lower=q_l))
    ).unstack('Status')

for status in y.columns:
    if status=='HIGH_CONFIDENCE': c='tab:blue'
    else: c='tab:orange'
    ax[1,0].plot(y.index, y[status], label=status.split('_')[0]+' ±std')
    ax[1,0].fill_between(y.index, y[status]-y_int[status], y[status]+y_int[status], color=c, alpha=0.1)
ax[1,0].legend()    
ax[1,0].set_ylabel('cBBI [ms] + grouped into 24 hrs') 
ax[1,0].set_xlabel('Hour of the day')
ax[1,0].set_ylim((450,1250))


In [ ]:
# lets do all users
upper_clip=0.999
fig,ax=plt.subplots(nrows=1, ncols=1, figsize=(10,6))
hr=pd.to_datetime(df['timeLocal_ms'], unit='ms').dt.hour 

y=df.groupby([hr, 'Status'])['BBI (ms)']\
    .agg(lambda s: np.mean(s.clip(upper=q_h, lower=q_l))).unstack('Status')
y_int=df.groupby([hr, 'Status'])['BBI (ms)']\
    .agg(lambda s: np.std(s.clip(upper=q_h, lower=q_l))).unstack('Status')

ax.plot(y.index, y['HIGH_CONFIDENCE'], linewidth=3, linestyle='solid', c='tab:blue', label='High confidence')
ax.plot(y.index, y['LOW_CONFIDENCE'], linewidth=3, linestyle='dashed', c='tab:orange', label='Low confidence')
ax.fill_between(y.index, y['HIGH_CONFIDENCE']-y_int['HIGH_CONFIDENCE'], y['HIGH_CONFIDENCE']+y_int['HIGH_CONFIDENCE'], color='tab:blue', alpha=0.075)
ax.fill_between(y.index, y['LOW_CONFIDENCE']-y_int['LOW_CONFIDENCE'], y['LOW_CONFIDENCE']+y_int['LOW_CONFIDENCE'], color='tab:orange', alpha=0.075)

# plot individual user lines
for user in df['User Id'].drop_duplicates():
    temp=df.loc[df['User Id']==user, :]
    q_h=temp['BBI (ms)'].quantile(upper_clip)
    q_l=temp['BBI (ms)'].quantile(1-upper_clip)

    y=temp.groupby([hr[temp.index], 'Status'])['BBI (ms)']\
        .agg(lambda s: np.mean(s.clip(upper=q_h, lower=q_l))).unstack('Status')
    for status in y.columns:
        if status=='HIGH_CONFIDENCE': 
            linestyle='solid'
            c='tab:blue'
        else: 
            linestyle=(5, (10, 3))
            c='tab:orange'
        ax.plot(y.index, y[status], linewidth=0.3, linestyle=linestyle, c=c)


ax.legend()    
ax.set_ylabel('cBBI [ms]') 
ax.set_xlabel('Hour') 
ax.set_ylim((450,1250))
ax.set_title(f'Average interbeat interval ±std\nall users and individual')
    
